# DoubleTM Voltage Curves

Build voltage curves from `doubleTM_dft_2nd_relax_summary.csv` for each `tm_pair`.
Each `elec_*` is treated as a distinct ordering.

In [1]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import ConvexHull
from pymatgen.core.composition import Composition
from pymatgen.core.lattice import Lattice

E_Na_ref = -1.31
F = 96485.33212
EXCLUDE_ORDERINGS = {}
PHASE_CHANGE_EDGE_WIDTH = 0.02

experiment_map = {
    "Mn_Fe": {
        "file": "NM2012_NMF.csv",
        "label": "Exp (Yabuuchi et al. Nat. Mater. 2012)",
        "is_capacity": True,
    },
    "Fe_Ni": {
        "file": "JMCA2017_NFN.csv",
        "label": "Exp (Vassilaras et al. JMCA 2017)",
        "is_capacity": False,
    },
    "Co_Ni": {
        "file": "JMCA2017_NCN.csv",
        "label": "Exp (Vassilaras et al. JMCA 2017)",
        "is_capacity": False,
    },
    "Mn_Ni": {
        "file": "IC2012_NMN.csv",
        "label": "Exp (Komaba et al. Inorg. Chem. 2012)",
        "is_capacity": False,
    },
    "Ti_Ni": {
        "file": "CM2016_NTN.csv",
        "label": "Exp (Nanba et al. Chem. Mater. 2016)",
        "is_capacity": False,
    },
    "Fe_Co": {
        "file": "AFM2016_NFC.csv",
        "label": "Exp (Kubota et al. AFM 2016)",
        "is_capacity": False,
    },
}

plt.rcParams.update({
    "font.size": 15,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
    "legend.title_fontsize": 15,
})

na_map = {
    "AJT_a": 1/7, "AJT_b": 1/4, "AJT_c": 1/4, "AJT_d": 1/3, "AJT_e": 1/3, 
    "AJT_f": 2/5, "AJT_g": 1/2, "AJT_h": 3/5, "AJT_i": 5/8, "AJT_j": 5/8,
    "AJT_k": 2/3, "AJT_l": 2/3, "AJT_m": 2/3, "AJT_n": 3/4, "AJT_o": 3/4,
    "AJT_p": 5/6, "AJT_q": 7/8, "AJT_r": 8/9,
    "JLK_b": 3/8, "JLK_c": 7/10, "JLK_theta_4": 4/5, "JLK_theta_5": 5/6,
    "JLK_delta": 2/3, "JLK_eta": 4/7, "JLK_zeta": 1/2, "JLK_zeta2": 2/5,
    "JLK_zeta3": 3/7, "JLK_zeta6": 6/11,
    "CD_a": 1/2, "JMAS_a": 1/3, "JMAS_b": 2/5, "JMAS_e": 1, "XC_a": 1/3,
    "XC_b": 1/2, "JMAS_c": 1/2, "JMAS_d": 2/3, "XL_a": 5/8
}

def load_csvs():
    sources = [
        (Path("../data/doubleTM_dft_tighter.csv"), "dft_summary"),
       # (Path("../data/doubleTM_dft_summary.csv"), "dft_summary"),
       # (Path("../data/doubleTM_dft_2nd_relax_summary.csv"), "dft_2nd_relax_summary"),
    ]
    frames = []
    for p, tag in sources:
        if p.exists():
            d = pd.read_csv(p)
            d["source_tag"] = tag
            frames.append(d)
            print(f"Loaded: {p} ({len(d)} rows)")
    if not frames:
        raise FileNotFoundError("No doubleTM summary CSV found.")
    return pd.concat(frames, ignore_index=True)

def robust_load_csv(path: Path) -> pd.DataFrame:
    clean_rows = []
    if not Path(path).exists():
        return pd.DataFrame(columns=[0, 1])

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("[") or line.startswith("#") or "source" in line.lower():
                continue
            parts = line.split(",") if "," in line else line.split()
            if len(parts) < 2:
                continue
            try:
                clean_rows.append((float(parts[0]), float(parts[1])))
            except ValueError:
                continue
    return pd.DataFrame(clean_rows, columns=[0, 1])

def lattice_c_from_cell(cell):
    if cell is None:
        return np.nan
    if isinstance(cell, float) and np.isnan(cell):
        return np.nan

    try:
        if isinstance(cell, dict):
            d = cell
        else:
            d = ast.literal_eval(str(cell))
        lat = Lattice.from_dict(d)
        return float(lat.c)
    except Exception:
        return np.nan

def short_name(ordering: str) -> str:
    s = str(ordering).replace("_w_vac", "")
    if s.startswith(("O1_", "O3_", "P3_")):
        rest = s.split("_", 1)[1]
        if rest in {"full", "empty"}:
            return rest
    return s

def phase_kind(phase_str: str) -> str:
    s = str(phase_str)
    if "O3" in s:
        return "O3"
    if "P3" in s:
        return "P3"
    if "O1" in s:
        return "O1"
    return "other"

def theoretical_capacity_full_mAh_g(tm_pair: str) -> float:
    tm1, tm2 = tm_pair.split("_")
    comp = Composition(f"Na1{tm1}0.5{tm2}0.5O2")
    M = comp.weight
    return F / 3.6 / M

def energy_per_TMO2(row):
    formula = row["formula"]
    natoms = float(row["natoms"])
    tm_pair = str(row["tm_pair"])
    tm_list = [x for x in tm_pair.split("_") if x]

    if not tm_list or pd.isna(formula) or pd.isna(natoms) or natoms <= 0:
        return np.nan

    comp_norm = Composition(formula)
    n_norm = comp_norm.num_atoms
    if n_norm <= 0:
        return np.nan

    scale = natoms / n_norm
    n_tmo2_units = sum(comp_norm[tm] for tm in tm_list) * scale
    if n_tmo2_units <= 0:
        return np.nan

    if "energy_eV" in row and not pd.isna(row["energy_eV"]):
        e_tot = float(row["energy_eV"])
    elif "energy_per_atom_eV" in row and not pd.isna(row["energy_per_atom_eV"]):
        e_tot = float(row["energy_per_atom_eV"]) * natoms
    else:
        return np.nan

    return e_tot / n_tmo2_units

def volume_per_tm(row):
    formula = row["formula"]
    natoms = float(row["natoms"])
    tm_pair = str(row["tm_pair"])
    tm_list = [x for x in tm_pair.split("_") if x]

    if not tm_list or pd.isna(formula) or pd.isna(natoms) or natoms <= 0:
        return np.nan

    if "volume_A3" not in row or pd.isna(row["volume_A3"]):
        return np.nan

    comp_norm = Composition(formula)
    n_norm = comp_norm.num_atoms
    if n_norm <= 0:
        return np.nan

    scale = natoms / n_norm
    n_tm_cell = sum(comp_norm[tm] for tm in tm_list) * scale
    if n_tm_cell <= 0:
        return np.nan

    return float(row["volume_A3"]) / n_tm_cell

def lower_hull_indices_scipy(xs, ys):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    if len(xs) == 0:
        return []

    unique_indices = []
    for x0 in sorted(set(xs)):
        idxs = np.where(np.isclose(xs, x0))[0]
        unique_indices.append(idxs[np.argmin(ys[idxs])])

    xs_u = xs[unique_indices]
    ys_u = ys[unique_indices]
    if len(xs_u) <= 2:
        return sorted(unique_indices, key=lambda i: xs[i])

    pts = np.column_stack([xs_u, ys_u])
    hull = ConvexHull(pts)
    verts_sorted = sorted(hull.vertices, key=lambda i: (xs_u[i], ys_u[i]))

    lower_u = []
    for i in verts_sorted:
        while len(lower_u) >= 2:
            j = lower_u[-2]
            k = lower_u[-1]
            x1, y1 = xs_u[j], ys_u[j]
            x2, y2 = xs_u[k], ys_u[k]
            x3, y3 = xs_u[i], ys_u[i]
            cross = (x2 - x1) * (y3 - y1) - (y2 - y1) * (x3 - x1)
            if cross <= 0:
                lower_u.pop()
            else:
                break
        lower_u.append(i)

    lower_idx = [unique_indices[i] for i in lower_u]
    return sorted(lower_idx, key=lambda i: xs[i])

df = load_csvs()

mask = pd.Series(True, index=df.index)
if "status" in df.columns:
    mask &= df["status"] == "OK"
if "n_custodian_errors" in df.columns:
    mask &= pd.to_numeric(df["n_custodian_errors"], errors="coerce").fillna(0) == 0
if "converged" in df.columns:
    c = df["converged"]
    if c.dtype == object:
        c = c.astype(str).str.lower() == "true"
    mask &= c
df = df[mask].copy()

df["natoms"] = pd.to_numeric(df["natoms"], errors="coerce")
df["energy_eV"] = pd.to_numeric(df.get("energy_eV"), errors="coerce")
df["volume_A3"] = pd.to_numeric(df.get("volume_A3"), errors="coerce")
if "energy_per_atom_eV" in df.columns:
    df["energy_per_atom_eV"] = pd.to_numeric(df["energy_per_atom_eV"], errors="coerce")

df["ordering_short"] = df["ordering"].apply(short_name)
if EXCLUDE_ORDERINGS:
    df = df[~df["ordering_short"].isin(EXCLUDE_ORDERINGS)].copy()

df["xNa"] = df["ordering_short"].map(na_map)
mask_full = df["ordering_short"] == "full"
mask_empty = df["ordering_short"] == "empty"
df.loc[mask_full, "xNa"] = 1.0
df.loc[mask_empty, "xNa"] = 0.0
df = df.dropna(subset=["xNa"]).copy()

df["E_TMO2_eV"] = df.apply(energy_per_TMO2, axis=1)
df["vol_per_tm_A3"] = df.apply(volume_per_tm, axis=1)
df["lattice_c_A"] = df["lattice"].apply(lattice_c_from_cell)
df = df.dropna(subset=["E_TMO2_eV", "lattice_c_A"]).copy()

elec_col = df["elec"].fillna("elec_unknown").astype(str)
src_col = df["source_tag"].fillna("unknown_src").astype(str)
df["ordering_elec"] = df["ordering_short"].astype(str) + "__" + elec_col + "__" + src_col

tm_pairs = sorted(df["tm_pair"].dropna().unique())
out_paths = []
for tm_pair in tm_pairs:
    df_tm = df[df["tm_pair"] == tm_pair].copy()
    if df_tm.empty:
        continue

    df_pts = (
        df_tm
        .groupby(["ordering_elec", "phase", "xNa"], as_index=False)
        .agg({"E_TMO2_eV": "mean", "vol_per_tm_A3": "mean", "lattice_c_A": "mean"})
    )
    if len(df_pts) < 2:
        continue

    xs = df_pts["xNa"].to_numpy(dtype=float)
    ys = df_pts["E_TMO2_eV"].to_numpy(dtype=float)
    hull_idx = lower_hull_indices_scipy(xs, ys)
    if len(hull_idx) < 2:
        continue

    hull_idx = sorted(hull_idx, key=lambda i: xs[i])
    x_hull = np.array([xs[i] for i in hull_idx], dtype=float)
    e_hull = np.array([ys[i] for i in hull_idx], dtype=float)
    vol_hull = np.array([df_pts.iloc[i]["vol_per_tm_A3"] for i in hull_idx], dtype=float)
    c_hull = np.array([df_pts.iloc[i]["lattice_c_A"] for i in hull_idx], dtype=float)
    phase_hull = [str(df_pts.iloc[i]["phase"]) for i in hull_idx]

    x_plot = []
    v_plot = []
    phase_regions = {"O3": [], "P3": [], "O1": []}
    for i in range(len(x_hull) - 1):
        x1, x2 = x_hull[i], x_hull[i + 1]
        if np.isclose(x2, x1):
            continue
        mu_na = (e_hull[i + 1] - e_hull[i]) / (x2 - x1)
        V = -(mu_na - E_Na_ref)
        x_plot.extend([x1, x2])
        v_plot.extend([V, V])

        kind_left = phase_kind(phase_hull[i])
        kind_right = phase_kind(phase_hull[i + 1])

        # If phase is unchanged across this segment, color the full span.
        if kind_left == kind_right and kind_left in phase_regions:
            phase_regions[kind_left].append((x1, x2))
            continue

        # For phase transitions, color only small edge bands near each endpoint,
        # leaving the middle region white.
        edge = min(PHASE_CHANGE_EDGE_WIDTH, 0.5 * (x2 - x1))
        if kind_left in phase_regions and edge > 0:
            phase_regions[kind_left].append((x1, x1 + edge))
        if kind_right in phase_regions and edge > 0:
            phase_regions[kind_right].append((x2 - edge, x2))

    if len(x_plot) < 2:
        continue

    exp_cfg = experiment_map.get(tm_pair)
    has_exp = False
    x_exp_plot = np.array([])
    v_exp_plot = np.array([])
    if exp_cfg is not None:
        exp_path = Path("../data/experiments") / exp_cfg["file"]
        df_exp = robust_load_csv(exp_path)
        if not df_exp.empty:
            raw_x = df_exp[0].to_numpy(dtype=float)
            raw_v = df_exp[1].to_numpy(dtype=float)
            if exp_cfg.get("is_capacity", False):
                C_full_for_exp = theoretical_capacity_full_mAh_g(tm_pair)
                x_conv = 1.0 - (raw_x / C_full_for_exp)
            else:
                x_conv = raw_x

            valid = np.isfinite(x_conv) & np.isfinite(raw_v)
            x_conv = x_conv[valid]
            raw_v = raw_v[valid]
            if len(x_conv) > 0:
                ordered = sorted(zip(x_conv, raw_v), key=lambda p: p[0])
                x_exp_plot = np.array([p[0] for p in ordered], dtype=float)
                v_exp_plot = np.array([p[1] for p in ordered], dtype=float)
                has_exp = True

    fig, ax = plt.subplots(figsize=(6, 4.5))
    phase_colors = {"O3": "#ff4d4f", "P3": "#1d8cff", "O1": "#22c55e"}
    eps = 1e-6
    for phase_kind_target, regions in phase_regions.items():
        if not regions:
            continue
        regions_sorted = sorted(regions, key=lambda s: s[0])
        merged_spans = []
        cur_start, cur_end = regions_sorted[0]
        for x_start, x_end in regions_sorted[1:]:
            if x_start <= cur_end + eps:
                cur_end = max(cur_end, x_end)
            else:
                merged_spans.append((cur_start, cur_end))
                cur_start, cur_end = x_start, x_end
        merged_spans.append((cur_start, cur_end))
        for x_start, x_end in merged_spans:
            ax.axvspan(x_start, x_end, facecolor=phase_colors[phase_kind_target], alpha=0.20, zorder=0)
    tm1, tm2 = tm_pair.split("_")
    ax.plot(x_plot, v_plot, color="black", linewidth=2.5, label="DFT (Tight Conv.)")
    ax_c = ax.twinx()
    valid_c = np.isfinite(c_hull)
    if np.any(valid_c):
        ax_c.plot(
            x_hull[valid_c],
            c_hull[valid_c],
            color="tab:gray",
            marker="s",
            markersize=10.0,
            markeredgecolor="black",
            markeredgewidth=1.4,
            linewidth=1.8,
            linestyle=(0, (3.5, 2.2)),
            alpha=0.9,
            label="C-axis length (Å)",
        )
    ax_c.set_ylabel("C-axis length (Å)", fontsize=20)
    ax_c.set_ylim(13, 18)
    ax_c.tick_params(axis="y", labelsize=18)
    if has_exp:
        ax.plot(x_exp_plot, v_exp_plot, color="tab:red", linestyle=(0, (3, 2)), linewidth=2.2, alpha=0.95, label=exp_cfg["label"])
    """
    ax.text(
        0.02,
        0.58,
        r"$\bf{Red:\ O3\ stable}$",
        transform=ax.transAxes,
        ha="left",
        va="center",
        fontsize=10,
        color="lightcoral",
    )
    ax.text(
        0.02,
        0.52,
        r"$\bf{Blue:\ P3\ stable}$",
        transform=ax.transAxes,
        ha="left",
        va="center",
        fontsize=10,
        color="deepskyblue",
    )
    ax.text(
        0.02,
        0.46,
        r"$\bf{Green:\ O1\ stable}$",
        transform=ax.transAxes,
        ha="left",
        va="center",
        fontsize=10,
        color="mediumseagreen",
    )
    """
    ax.set_xlabel(f"x in Na$_{{\\mathrm{{x}}}}${tm1}$_{{0.5}}${tm2}$_{{0.5}}$O$_2$", fontsize=20)
    ax.set_ylabel("Voltage (V vs Na/Na$^+$)", fontsize=20)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0, 6)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.tick_params(axis="both", labelsize=18)
    handles_l, labels_l = ax.get_legend_handles_labels()
    handles_r, labels_r = ax_c.get_legend_handles_labels()
    ax.legend(
        handles_l + handles_r,
        labels_l + labels_r,
        loc="lower right",
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.78,
        edgecolor="black",
        facecolor="white",
    )

    ax_top = ax.twiny()
    x_min, x_max = ax.get_xlim()
    ax_top.set_xlim(x_max, x_min)
    C_full = theoretical_capacity_full_mAh_g(tm_pair)
    ax_top.set_xlabel("Capacity (mAh/g)", fontsize=20)
    ticks = np.arange(0, C_full + 0.1, 50)
    if len(ticks) > 1 and abs(ticks[-1] - C_full) < 10:
        ticks = ticks[:-1]
    ticks = list(ticks) + [C_full]
    ax_top.set_xticks([1.0 - (t / C_full) for t in ticks])
    ax_top.set_xticklabels([f"{t:.0f}" for t in ticks])
    ax_top.tick_params(axis="x", labelsize=18)

    outpath = Path(f"../data/figures/doubleTM/voltage_tighterConv_{tm_pair}.pdf")
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    out_paths.append(str(outpath))

print(f"Saved {len(out_paths)} voltage plots")
out_paths[:10]


Loaded: ../data/doubleTM_dft_tighter.csv (820 rows)
Saved 10 voltage plots


['../data/figures/doubleTM/voltage_tighterConv_Co_Ni.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_Fe_Co.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_Fe_Ni.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_Mn_Co.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_Mn_Fe.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_Mn_Ni.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_Ti_Co.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_Ti_Fe.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_Ti_Mn.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_Ti_Ni.pdf']

In [2]:
# Voltage curve with magnetic moments (same voltage pipeline as previous cell)
import ast

if "df" not in globals():
    raise RuntimeError("Run the previous setup/voltage cell first so `df` and helpers are available.")

if "experiment_map" not in globals():
    raise RuntimeError("Run the previous setup/voltage cell first so `experiment_map` is available.")


def avg_abs_tm_magmom(tm_magmom_dict_str, tm_symbol):
    if pd.isna(tm_magmom_dict_str):
        return np.nan
    try:
        parsed = ast.literal_eval(tm_magmom_dict_str)
    except Exception:
        return np.nan

    if not isinstance(parsed, dict) or tm_symbol not in parsed:
        return np.nan

    try:
        arr = np.array(parsed[tm_symbol], dtype=float)
    except Exception:
        return np.nan

    if arr.size == 0:
        return np.nan

    return float(np.mean(np.abs(arr)))


def tm_magmom_list(tm_magmom_dict_str, tm_symbol):
    if pd.isna(tm_magmom_dict_str):
        return []
    try:
        parsed = ast.literal_eval(tm_magmom_dict_str)
    except Exception:
        return []
    if not isinstance(parsed, dict) or tm_symbol not in parsed:
        return []
    vals = parsed[tm_symbol]
    if not isinstance(vals, (list, tuple, np.ndarray)):
        return []
    out = []
    for v in vals:
        try:
            out.append(float(v))
        except Exception:
            pass
    return out


# Read merged magmom table (elec_0 + elec_1)
magmom_df = pd.read_csv("../data/doubleTM_magmom.csv")

# Exact lookup key requested by user: (phase, ordering, tm_pair, elec)
mm_lookup = {
    (row["phase"], row["ordering"], row["tm_pair"], row["tm_ordering"]): row["tm_magmom_dict"]
    for _, row in magmom_df.iterrows()
}


tm_pairs = sorted(df["tm_pair"].dropna().unique())
out_paths_magmom = []
stable_point_tables = []

for tm_pair in tm_pairs:
    df_tm = df[df["tm_pair"] == tm_pair].copy()
    if df_tm.empty:
        continue

    # --- Keep same point construction as previous voltage cell ---
    df_pts = (
        df_tm
        .groupby(["ordering_elec", "phase", "xNa"], as_index=False)
        .agg({
            "E_TMO2_eV": "mean",
            "vol_per_tm_A3": "mean",
            "lattice_c_A": "mean",
            "ordering": "first",
            "elec": "first",
        })
    )
    if len(df_pts) < 2:
        continue

    xs = df_pts["xNa"].to_numpy(dtype=float)
    ys = df_pts["E_TMO2_eV"].to_numpy(dtype=float)
    hull_idx = lower_hull_indices_scipy(xs, ys)
    if len(hull_idx) < 2:
        continue

    hull_idx = sorted(hull_idx, key=lambda i: xs[i])
    x_hull = np.array([xs[i] for i in hull_idx], dtype=float)
    e_hull = np.array([ys[i] for i in hull_idx], dtype=float)
    phase_hull = [str(df_pts.iloc[i]["phase"]) for i in hull_idx]

    tm1, tm2 = tm_pair.split("_")

    # Match magnetic moments only for stable compounds with exact key
    m1_hull = []
    m2_hull = []
    matched = []
    stable_rows = []

    for i in hull_idx:
        row = df_pts.iloc[i]
        key = (str(row["phase"]), str(row["ordering"]), str(tm_pair), str(row["elec"]))
        mm_str = mm_lookup.get(key, np.nan)

        list_m1 = tm_magmom_list(mm_str, tm1)
        list_m2 = tm_magmom_list(mm_str, tm2)
        m1 = float(np.mean(np.abs(list_m1))) if len(list_m1) > 0 else np.nan
        m2 = float(np.mean(np.abs(list_m2))) if len(list_m2) > 0 else np.nan
        is_match = np.isfinite(m1) and np.isfinite(m2)

        m1_hull.append(m1)
        m2_hull.append(m2)
        matched.append(is_match)

        stable_rows.append({
            "tm_pair": tm_pair,
            "phase": row["phase"],
            "ordering": row["ordering"],
            "elec": row["elec"],
            "xNa": row["xNa"],
            f"magmom_list_{tm1}": list_m1,
            f"magmom_list_{tm2}": list_m2,
            f"avg_abs_magmom_{tm1}": m1,
            f"avg_abs_magmom_{tm2}": m2,
            "matched_magmom": is_match,
        })

    m1_hull = np.array(m1_hull, dtype=float)
    m2_hull = np.array(m2_hull, dtype=float)
    stable_point_tables.append(pd.DataFrame(stable_rows))

    # --- Keep same voltage segment construction as previous cell ---
    V_segments = []
    phase_regions = {"O3": [], "P3": [], "O1": []}
    for i in range(len(x_hull) - 1):
        x1, x2 = x_hull[i], x_hull[i + 1]
        if np.isclose(x2, x1):
            continue

        mu_na = (e_hull[i + 1] - e_hull[i]) / (x2 - x1)
        V_seg = -(mu_na - E_Na_ref)
        V_segments.append((x1, x2, V_seg))

        kind_left = phase_kind(phase_hull[i])
        kind_right = phase_kind(phase_hull[i + 1])

        if kind_left == kind_right and kind_left in phase_regions:
            phase_regions[kind_left].append((x1, x2))
            continue

        edge = min(PHASE_CHANGE_EDGE_WIDTH, 0.5 * (x2 - x1))
        if kind_left in phase_regions and edge > 0:
            phase_regions[kind_left].append((x1, x1 + edge))
        if kind_right in phase_regions and edge > 0:
            phase_regions[kind_right].append((x2 - edge, x2))

    if not V_segments:
        continue

    has_exp = False
    x_exp_plot, v_exp_plot = [], []
    exp_cfg = experiment_map.get(tm_pair)
    if exp_cfg is not None:
        exp_path = Path(f"../data/experiments/{exp_cfg['file']}")
        df_exp = robust_load_csv(exp_path)
        if not df_exp.empty:
            raw_x = df_exp[0].values
            raw_v = df_exp[1].values
            C_theo = theoretical_capacity_full_mAh_g(tm_pair)
            max_val = np.max(raw_x) if len(raw_x) else 0.0

            if exp_cfg["is_capacity"] and max_val < 5.0:
                x_conv = raw_x
            elif (not exp_cfg["is_capacity"]) and max_val > 5.0:
                x_conv = 1.0 - (raw_x / C_theo)
            elif exp_cfg["is_capacity"]:
                x_conv = 1.0 - (raw_x / C_theo)
            else:
                x_conv = raw_x

            combined = sorted(zip(x_conv, raw_v), key=lambda p: p[0])
            if combined:
                x_exp_plot = np.array([p[0] for p in combined])
                v_exp_plot = np.array([p[1] for p in combined])
                has_exp = True

    fig, ax = plt.subplots(figsize=(6, 4.5))

    phase_colors = {"O3": "#ff4d4f", "P3": "#1d8cff", "O1": "#22c55e"}
    eps = 1e-6
    for phase_kind_target, regions in phase_regions.items():
        if not regions:
            continue
        regions_sorted = sorted(regions, key=lambda seg: seg[0])
        merged_spans = []
        cur_start, cur_end = regions_sorted[0]
        for x_start, x_end in regions_sorted[1:]:
            if x_start <= cur_end + eps:
                cur_end = max(cur_end, x_end)
            else:
                merged_spans.append((cur_start, cur_end))
                cur_start, cur_end = x_start, x_end
        merged_spans.append((cur_start, cur_end))
        for x_start, x_end in merged_spans:
            ax.axvspan(x_start, x_end, facecolor=phase_colors[phase_kind_target], alpha=0.20, zorder=0)

    x_plot = []
    v_plot = []
    for x1, x2, v in V_segments:
        x_plot.extend([x1, x2])
        v_plot.extend([v, v])

    ax.plot(x_plot, v_plot, color="black", linewidth=2.5, label="DFT (Tight Conv.)")

    # Plot only matched stable compounds; unmatched stable points are skipped
    ax_mm = ax.twinx()
    valid_m1 = np.isfinite(m1_hull)
    valid_m2 = np.isfinite(m2_hull)

    if np.any(valid_m1):
        ax_mm.plot(
            x_hull[valid_m1],
            m1_hull[valid_m1],
            color="#0b3d91",
            marker="D",
            markersize=10.0,
            markeredgecolor="black",
            markeredgewidth=1.0,
            linewidth=1.6,
            linestyle=(0, (1.2, 2.2)),
            alpha=0.95,
            label=f"Avg of Magnetic Moment ({tm1})",
            zorder=3,
        )

    if np.any(valid_m2):
        ax_mm.plot(
            x_hull[valid_m2],
            m2_hull[valid_m2],
            color="#8b1e3f",
            marker="D",
            markersize=10.0,
            markeredgecolor="black",
            markeredgewidth=1.0,
            linewidth=1.6,
            linestyle=(0, (1.2, 2.2)),
            alpha=0.95,
            label=f"Avg of Magnetic Moment ({tm2})",
            zorder=3,
        )

    ax_mm.set_ylabel(r"Avg of Magnetic Moment ($\mu_B$)", fontsize=20)
    ax_mm.set_ylim(0, 5)
    ax_mm.tick_params(axis="y", labelsize=18)

    if has_exp:
        ax.plot(
            x_exp_plot,
            v_exp_plot,
            color="tab:red",
            linestyle=(0, (3, 2)),
            linewidth=2.2,
            alpha=0.95,
            label=exp_cfg["label"],
        )

    ax.set_xlabel(f"x in Na$_{{\\mathrm{{x}}}}${tm1}$_{{0.5}}${tm2}$_{{0.5}}$O$_2$", fontsize=20)
    ax.set_ylabel("Voltage (V vs Na/Na$^+$)", fontsize=20)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0, 6)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.tick_params(axis="both", labelsize=18)

    handles_l, labels_l = ax.get_legend_handles_labels()
    handles_r, labels_r = ax_mm.get_legend_handles_labels()
    ax.legend(
        handles_l + handles_r,
        labels_l + labels_r,
        loc="center left",
        bbox_to_anchor=(1.12, 0.5),
        borderaxespad=0.0,
        fontsize=11,
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.78,
        edgecolor="black",
        facecolor="white",
    )

    ax_top = ax.twiny()
    x_min, x_max = ax.get_xlim()
    ax_top.set_xlim(x_max, x_min)
    C_full = theoretical_capacity_full_mAh_g(tm_pair)
    ax_top.set_xlabel("Capacity (mAh/g)", fontsize=20)
    ticks = np.arange(0, C_full + 0.1, 50)
    if len(ticks) > 1 and abs(ticks[-1] - C_full) < 10:
        ticks = ticks[:-1]
    ticks = list(ticks) + [C_full]
    ax_top.set_xticks([1.0 - (t / C_full) for t in ticks])
    ax_top.set_xticklabels([f"{t:.0f}" for t in ticks])
    ax_top.tick_params(axis="x", labelsize=18)

    outpath = Path(f"../data/figures/doubleTM/voltage_tighterConv_magmom_{tm_pair}.pdf")
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    out_paths_magmom.append(str(outpath))

stable_points_magmom_df = pd.concat(stable_point_tables, ignore_index=True)
stable_points_magmom_df = stable_points_magmom_df.sort_values(["tm_pair", "xNa"]).reset_index(drop=True)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(stable_points_magmom_df)

missing = stable_points_magmom_df[~stable_points_magmom_df["matched_magmom"]]
if not missing.empty:
    print("Note: unmatched stable compounds were skipped for magnetic-moment plotting:")
    display(missing[["tm_pair", "phase", "ordering", "elec", "xNa"]].sort_values(["tm_pair", "xNa"]).reset_index(drop=True))

print(f"Saved {len(out_paths_magmom)} voltage+magmom plots")
out_paths_magmom[:10]



,tm_pair,phase,ordering,elec,xNa,magmom_list_Co,magmom_list_Ni,avg_abs_magmom_Co,avg_abs_magmom_Ni,matched_magmom,magmom_list_Fe,avg_abs_magmom_Fe,magmom_list_Mn,avg_abs_magmom_Mn,magmom_list_Ti,avg_abs_magmom_Ti
0,Co_Ni,O3_orderings,O3_empty,elec_1,0.000000,"[1.109, 1.109, 1.109, 1.109, 1.109, 1.109]","[0.092, 0.092, 0.092, 0.092, 0.092, 0.092]",1.109000,0.092000,True,NaN,NaN,NaN,NaN,NaN,NaN
1,Co_Ni,P3_orderings,JLK_zeta2_w_vac,elec_0,0.400000,"[-0.031, -0.01, 1.077, 0.001, -1.057, -0.009, -0.059, -0.007, 1.075, -1.013, 1.051, 0.017, 0.015, 1.063, -0.024]","[0.129, 0.08, 0.228, 1.106, 0.081, 0.029, -0.002, 1.101, 0.025, 0.033, 0.051, 0.033, 1.103, 0.076, 0.062]",0.433933,0.275933,True,NaN,NaN,NaN,NaN,NaN,NaN
2,Co_Ni,P3_orderings,JLK_zeta3_w_vac,elec_0,0.428571,"[0.996, 0.085, 0.086, 1.037, -1.071, 0.015, 0.014, 0.061, 0.041, 0.028, 1.053, 0.983, 0.093, 0.99, -1.065, 1.048, -0.022, -0.003, -0.051, -0.011, -0.024]","[-1.133, -1.157, -0.18, -0.141, -0.12, -0.207, -0.288, 1.141, 0.05, 0.106, 0.053, 0.245, 0.299, 0.013, -0.05, -1.199, -1.204, -0.173, -0.075, -0.149, -0.082]",0.417952,0.384048,True,NaN,NaN,NaN,NaN,NaN,NaN
3,Co_Ni,P3_orderings,JLK_zeta_w_vac,elec_0,0.500000,"[1.091, 1.091, 0.161, 0.97, -0.069, -0.995]","[-0.96, -1.063, 0.139, 1.157, 1.113, 1.114]",0.729500,0.924333,True,NaN,NaN,NaN,NaN,NaN,NaN
4,Co_Ni,P3_orderings,JLK_delta_w_vac,elec_1,0.666667,"[1.116, 0.991, -0.065, 1.02, 1.097, -0.02, 1.086, 1.088, -0.959, -0.977, 1.009, 1.098, 1.094, 0.011, 1.031, -0.031, 0.004, 1.086, 1.024, 1.016, 1.057, -0.027, -0.026, 1.061, 1.08, 1.081, 1.046]","[2.06, 1.182, 1.13, 0.012, 1.173, 1.218, 1.141, 1.131, 1.759, 1.15, 1.75, 1.215, 1.166, 2.092, 1.755, 1.758, 0.096, -0.018, 1.739, 1.21, 1.155, 1.221, 0.231, 1.183, 1.216, 1.156, 1.73]",0.785222,1.209148,True,NaN,NaN,NaN,NaN,NaN,NaN
5,Co_Ni,O3_orderings,AJT_o_w_vac,elec_1,0.750000,"[-0.047, 2.378, 1.102, 1.102, -0.939, -0.942]","[1.29, 1.232, 1.425, 1.63, 1.734, 1.353]",1.085000,1.444000,True,NaN,NaN,NaN,NaN,NaN,NaN
6,Co_Ni,O3_orderings,JLK_theta_5_w_vac,elec_1,0.833333,"[3.008, 1.129, -0.79, 1.129, 3.009, -0.79, 1.091, 1.09, -1.047]","[1.121, 1.744, 1.602, 1.63, 1.699, 1.63, 1.603, 1.745, 1.121]",1.453667,1.543889,True,NaN,NaN,NaN,NaN,NaN,NaN
7,Co_Ni,O3_orderings,AJT_r_w_vac,elec_1,0.888889,"[3.042, 2.544, 2.71, 2.496, 3.049, 1.102, 1.108, -1.054, -0.971, -1.011, -0.958, 3.065, 3.047, -0.19, 3.083, -1.035, 1.172, 1.099, 1.117, 1.138, 1.124, 2.946, 1.13, -0.2, 2.971, -0.964, -0.07]","[1.72, 1.745, 1.297, 1.742, 1.754, 1.343, 1.33, 1.345, 1.732, 1.734, 1.649, 1.742, 1.438, 1.348, 1.699, 1.402, 1.23, 1.417, 1.724, 1.331, 1.509, -1.178, 1.278, 1.454, 1.323, 1.729, 1.695]",1.644296,1.514370,True,NaN,NaN,NaN,NaN,NaN,NaN
8,Co_Ni,P3_orderings,P3_full,elec_0,1.000000,"[1.091, 1.091, 1.091, 1.091, 1.091, 1.091]","[1.754, 1.754, 1.754, 1.754, 1.754, 1.754]",1.091000,1.754000,True,NaN,NaN,NaN,NaN,NaN,NaN
9,Fe_Co,P3_orderings,P3_empty,elec_1,0.000000,"[1.167, 1.167, -0.899, -0.899, -0.899, -0.899]",NaN,0.988333,NaN,True,"[3.981, 3.981, 3.936, 3.936, 3.936, 3.936]",3.951000,NaN,NaN,NaN,NaN


Saved 10 voltage+magmom plots


['../data/figures/doubleTM/voltage_tighterConv_magmom_Co_Ni.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_magmom_Fe_Co.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_magmom_Fe_Ni.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_magmom_Mn_Co.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_magmom_Mn_Fe.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_magmom_Mn_Ni.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_magmom_Ti_Co.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_magmom_Ti_Fe.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_magmom_Ti_Mn.pdf',
 '../data/figures/doubleTM/voltage_tighterConv_magmom_Ti_Ni.pdf']